# Riveting Process Intelligence: Head Diameter Prediction via Linear Regression

---
**Project 07 · LozanoLsa Industrial ML Portfolio**  
**Domain:** Automotive & Aerospace Manufacturing — Joining & Assembly  
**Algorithm:** Multiple Linear Regression (OLS + Ridge · Lasso · ElasticNet comparison)  
**Target:** `head_diameter_mm` — Formed rivet head diameter (continuous, mm)

---

## 🏭 Business Context

In high-volume riveting operations — automotive body panels, aerospace fuselage skins, HVAC ductwork — the formed head of a rivet must land within a tight specification window. Too small, and the joint lacks the bearing area for structural integrity. Too large, and the head interferes with adjacent parts or requires costly rework.

Operators traditionally calibrate press parameters empirically, relying on sample inspections after the fact. By the time the measuring system confirms a deviation, dozens of non-conforming parts have already been produced.

**This project replaces that reactive loop with a predictive one.** A linear regression model trained on 1,763 press cycles — covering six controllable and environmental process variables — estimates the final head diameter *before committing to production*. The engineering team can simulate the impact of changing rivet size, adjusting stroke, or switching to a different batch without touching the press.

The model is deliberately interpretable: every coefficient maps directly to an engineering lever, making it a tool that production engineers can understand, challenge, and trust.

---

*"The best process control is the one that tells you what will happen — not what already did."*


## 1 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, normaltest, probplot

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ── Colour palette (LozanoLsa)
C_BLUE  = "#4C9BE8"
C_RED   = "#E8574C"
C_GREEN = "#2ECC71"
C_AMBER = "#F39C12"
C_DARK  = "#1B2840"
C_MUTED = "#697888"

SPEC_LO, SPEC_HI = 5.0, 5.4   # head diameter spec window (mm)
SPEC_MID = (SPEC_LO + SPEC_HI) / 2   # 5.2 mm target centre

FEATURES = [
    "press_force_kn",
    "press_stroke_mm",
    "rivet_diameter_mm",
    "rivet_length_mm",
    "temperature_c",
    "hold_time_ms",
]
TARGET = "head_diameter_mm"

print("✓ Environment ready")


## 2 · Load Data

The dataset was extracted from the press PLC monitoring system and complemented with dimensional measurements captured by the inline CMM station at the end of the riveting cell. Each row represents one press cycle.

| Column | Type | Description |
|---|---|---|
| `press_force_kn` | float | Applied press force (kN) |
| `press_stroke_mm` | float | Ram travel distance (mm) |
| `rivet_diameter_mm` | float | Nominal shank diameter: 3, 4, or 5 mm |
| `rivet_length_mm` | float | Rivet shank length (mm) |
| `temperature_c` | float | Ambient temperature in press bay (°C) |
| `hold_time_ms` | float | Dwell time at full press force (ms) |
| `head_diameter_mm` | float | **Target** — measured formed head diameter (mm) |
| `head_diameter_ok` | int | 1 = within spec [5.0, 5.4] mm; 0 = out of spec |


In [ ]:
try:
    df = pd.read_csv("lr_riveting_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/LR_Cycle_Time_Estimation/main/lr_riveting_data.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


## 3 · Sanity Checks

In [ ]:
print("── Shape ──────────────────────")
print(f"  Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

print("\n── Data Types ─────────────────")
print(df.dtypes.to_string())

print("\n── Descriptive Statistics ─────")
display(df.describe().round(2).T)


In [ ]:
print("── Missing Values ──────────────")
nulls = df.isna().sum()
print(nulls[nulls > 0] if nulls.any() else "  ✓ No missing values detected.")

print("\n── Duplicate Rows ──────────────")
dups = df.duplicated().sum()
print(f"  {dups} duplicate rows." if dups else "  ✓ No duplicate rows.")

print("\n── Rivet Diameter Distribution ─")
vc = df["rivet_diameter_mm"].value_counts().sort_index()
for val, cnt in vc.items():
    print(f"  {val:.0f} mm: {cnt:,} rows ({cnt/len(df)*100:.1f}%)")

print("\n── Target Range ────────────────")
print(f"  head_diameter_mm: [{df[TARGET].min():.3f}, {df[TARGET].max():.3f}] mm")
print(f"  Mean: {df[TARGET].mean():.3f} mm  |  Std: {df[TARGET].std():.3f} mm")
pct_ok = df["head_diameter_ok"].mean() * 100
print(f"\n  Within spec [5.0 – 5.4 mm]: {pct_ok:.1f}% of cycles")


## 4 · Exploratory Data Analysis

With a continuous target variable, EDA serves two purposes: understand *where* the process is running relative to spec, and identify *which levers* have the strongest linear relationship with head diameter. We start at the target and work outward to the features.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── (A) Histogram with spec window
ax = axes[0]
ax.hist(df[TARGET], bins=50, color=C_BLUE, alpha=0.75, edgecolor="white", linewidth=0.4)
ax.axvspan(SPEC_LO, SPEC_HI, alpha=0.12, color=C_GREEN, label=f"Spec window [{SPEC_LO}–{SPEC_HI}] mm")
ax.axvline(df[TARGET].mean(), color=C_RED, ls="--", lw=1.5, label=f"Mean = {df[TARGET].mean():.3f} mm")
ax.axvline(SPEC_LO, color=C_GREEN, ls="--", lw=1.2)
ax.axvline(SPEC_HI, color=C_GREEN, ls="--", lw=1.2)
ax.set_xlabel("Head Diameter (mm)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Distribution of Head Diameter", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

# ── (B) Box plot by rivet diameter
ax2 = axes[1]
groups = [df[df["rivet_diameter_mm"] == d][TARGET].values for d in [3.0, 4.0, 5.0]]
bp = ax2.boxplot(groups, labels=["3 mm", "4 mm", "5 mm"], patch_artist=True,
                 medianprops=dict(color="white", linewidth=2))
colors = [C_RED, C_BLUE, C_AMBER]
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.75)
ax2.axhspan(SPEC_LO, SPEC_HI, alpha=0.10, color=C_GREEN, label="Spec window")
ax2.set_xlabel("Rivet Nominal Diameter", fontsize=11)
ax2.set_ylabel("Head Diameter (mm)", fontsize=11)
ax2.set_title("Head Diameter by Rivet Size", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)

plt.suptitle("Target Variable: head_diameter_mm", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Spec compliance by rivet size:")
for d in [3.0, 4.0, 5.0]:
    sub = df[df["rivet_diameter_mm"] == d]
    pct = sub["head_diameter_ok"].mean() * 100
    print(f"  {d:.0f} mm rivet: {pct:.1f}% within spec")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    colors = df["head_diameter_ok"].map({1: C_BLUE, 0: C_RED})
    ax.scatter(df[feat], df[TARGET], c=colors, alpha=0.3, s=8)
    # Best-fit line
    m, b, r, p, _ = stats.linregress(df[feat], df[TARGET])
    xr = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(xr, m * xr + b, color=C_DARK, lw=1.5, ls="--", label=f"r = {r:.3f}")
    ax.axhspan(SPEC_LO, SPEC_HI, alpha=0.07, color=C_GREEN)
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel(TARGET, fontsize=9)
    ax.set_title(f"{feat}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    # Colour legend only on first cell
    if i == 0:
        from matplotlib.lines import Line2D
        legend_els = [Line2D([0],[0], marker='o', color='w', markerfacecolor=C_BLUE, markersize=7, label='In-spec'),
                      Line2D([0],[0], marker='o', color='w', markerfacecolor=C_RED,  markersize=7, label='Out-of-spec')]
        ax.legend(handles=legend_els + [ax.get_lines()[0]], fontsize=7)

plt.suptitle("Process Variables vs Head Diameter (blue = in-spec, red = out-of-spec)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
corr = df[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # upper triangle only
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, linewidths=0.5, linecolor="white",
    cbar_kws={"shrink": 0.8}, ax=ax
)
ax.set_title("Pearson Correlation Matrix", fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

print("Correlation with head_diameter_mm (sorted):")
corr_target = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
for feat, val in corr_target.items():
    bar = "█" * int(abs(val) * 20)
    print(f"  {feat:<28}: {val:+.4f}  {bar}")


## 5 · Preprocessing

Linear regression does not require feature scaling when we care only about prediction accuracy.  
However, we fit **Ridge**, **Lasso**, and **ElasticNet** variants in a pipeline with `StandardScaler` so that regularisation penalties are applied on a common scale — making the comparison fair.

The `head_diameter_ok` column is an auxiliary label derived from the target; it is excluded from the feature set to prevent leakage.


In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set : {X_train.shape[0]:,} rows")
print(f"Test  set : {X_test.shape[0]:,}  rows")
print(f"Feature   : {FEATURES}")
print(f"Target    : {TARGET}")


## 6 · Model Training

We benchmark four regression variants to answer a single question: **does regularisation add value when the true relationship is nearly linear?**

| Model | Penalty | Expected behaviour |
|---|---|---|
| **OLS Linear Regression** | None | Unbiased; best when all features genuinely relevant |
| **Ridge (α = 1.0)** | L2 (sum of squares) | Shrinks all coefficients; stable under collinearity |
| **Lasso (α = 0.01)** | L1 (sum of absolute values) | Can zero out weak features; sparse solution |
| **ElasticNet (α = 0.01, ρ = 0.5)** | L1 + L2 mix | Balances sparsity and stability |

Ridge and ElasticNet are wrapped in a `StandardScaler` pipeline — regularisation penalties are scale-sensitive.


In [ ]:
# ── Metric helper ────────────────────────────────────────────────────
def regression_metrics(name, y_tr, yp_tr, y_te, yp_te):
    return {
        "Model"      : name,
        "MAE_train"  : round(mean_absolute_error(y_tr, yp_tr), 5),
        "MAE_test"   : round(mean_absolute_error(y_te, yp_te), 5),
        "RMSE_train" : round(np.sqrt(mean_squared_error(y_tr, yp_tr)), 5),
        "RMSE_test"  : round(np.sqrt(mean_squared_error(y_te, yp_te)), 5),
        "R2_train"   : round(r2_score(y_tr, yp_tr), 5),
        "R2_test"    : round(r2_score(y_te, yp_te), 5),
    }

results = []

# ── 1) OLS Linear Regression ─────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)
results.append(regression_metrics(
    "OLS Linear Regression",
    y_train, lr.predict(X_train),
    y_test,  lr.predict(X_test)
))

# ── 2) Ridge (L2) ────────────────────────────────────────────────────
ridge_pipe = Pipeline([("scaler", StandardScaler()), ("reg", Ridge(alpha=1.0))])
ridge_pipe.fit(X_train, y_train)
results.append(regression_metrics(
    "Ridge (α = 1.0)",
    y_train, ridge_pipe.predict(X_train),
    y_test,  ridge_pipe.predict(X_test)
))

# ── 3) Lasso (L1) ────────────────────────────────────────────────────
lasso_pipe = Pipeline([("scaler", StandardScaler()), ("reg", Lasso(alpha=0.01, max_iter=10_000))])
lasso_pipe.fit(X_train, y_train)
results.append(regression_metrics(
    "Lasso (α = 0.01)",
    y_train, lasso_pipe.predict(X_train),
    y_test,  lasso_pipe.predict(X_test)
))

# ── 4) ElasticNet (L1 + L2) ──────────────────────────────────────────
enet_pipe = Pipeline([("scaler", StandardScaler()),
                      ("reg", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10_000))])
enet_pipe.fit(X_train, y_train)
results.append(regression_metrics(
    "ElasticNet (α = 0.01, ρ = 0.5)",
    y_train, enet_pipe.predict(X_train),
    y_test,  enet_pipe.predict(X_test)
))

df_results = pd.DataFrame(results)
print("Model Comparison — Test Set")
display(df_results.style.format({
    "MAE_train" : "{:.5f}", "MAE_test" : "{:.5f}",
    "RMSE_train": "{:.5f}", "RMSE_test": "{:.5f}",
    "R2_train"  : "{:.5f}", "R2_test"  : "{:.5f}",
}).highlight_max(subset=["R2_test"], color="#d5f5e3")
  .highlight_min(subset=["RMSE_test"], color="#d5f5e3"))


## 7 · Model Evaluation

The OLS model is selected as the production model: it matches or marginally outperforms regularised variants on every metric, confirming that the six features carry genuine signal and regularisation adds no benefit here.

Key performance figures on the **held-out test set (n = 353)**:

| Metric | Value | Operational Meaning |
|---|---|---|
| **R²** | **0.909** | 90.9% of head-diameter variance explained by process inputs |
| **RMSE** | **0.081 mm** | Typical prediction error — 8.1% of the 1.0 mm spec window |
| **MAE** | **0.065 mm** | Median absolute miss — well within measurement system tolerance |
| **Baseline R²** | −0.011 | A naïve "predict the mean" model performs no better than chance |


In [ ]:
y_pred_test = lr.predict(X_test)
residuals   = y_test - y_pred_test

mae_test  = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test   = r2_score(y_test, y_pred_test)

print(f"OLS Linear Regression — Test Set Performance")
print(f"  R²   : {r2_test:.4f}")
print(f"  RMSE : {rmse_test:.4f} mm")
print(f"  MAE  : {mae_test:.4f} mm")
print(f"  Spec window width : {SPEC_HI - SPEC_LO:.1f} mm")
print(f"  RMSE / Spec width : {rmse_test / (SPEC_HI - SPEC_LO) * 100:.1f}%  (lower is better)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── (A) Predicted vs Real ──────────────────────────────────────────
ax = axes[0]
in_spec_mask = (y_test >= SPEC_LO) & (y_test <= SPEC_HI)
ax.scatter(y_test[in_spec_mask],  y_pred_test[in_spec_mask],
           alpha=0.5, s=12, color=C_BLUE,  label="In-spec")
ax.scatter(y_test[~in_spec_mask], y_pred_test[~in_spec_mask],
           alpha=0.5, s=12, color=C_RED,   label="Out-of-spec")
lims = [y_test.min() - 0.05, y_test.max() + 0.05]
ax.plot(lims, lims, "k--", lw=1.2, label="Perfect prediction")
ax.axvspan(SPEC_LO, SPEC_HI, alpha=0.06, color=C_GREEN)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Measured head_diameter_mm", fontsize=11)
ax.set_ylabel("Predicted head_diameter_mm", fontsize=11)
ax.set_title("Predicted vs Measured (Test Set)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.text(0.05, 0.92, f"R² = {r2_test:.4f}", transform=ax.transAxes, fontsize=10,
        color=C_DARK, fontweight="bold")

# ── (B) Residuals vs Fitted ────────────────────────────────────────
ax2 = axes[1]
ax2.scatter(y_pred_test, residuals, alpha=0.35, s=12,
            c=np.where(np.abs(residuals) > 0.15, C_RED, C_BLUE))
ax2.axhline(0, color=C_DARK, lw=1.5, ls="--")
ax2.axhline(+2 * rmse_test, color=C_AMBER, lw=1, ls=":", label=f"±2·RMSE = ±{2*rmse_test:.3f}")
ax2.axhline(-2 * rmse_test, color=C_AMBER, lw=1, ls=":")
ax2.set_xlabel("Predicted head_diameter_mm", fontsize=11)
ax2.set_ylabel("Residual (measured − predicted)", fontsize=11)
ax2.set_title("Residuals vs Fitted Values", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.text(0.05, 0.05, f"RMSE = {rmse_test:.4f} mm", transform=ax2.transAxes, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── (A) Residual Distribution ──────────────────────────────────────
ax = axes[0]
sns.histplot(residuals, bins=30, kde=True, color=C_BLUE, alpha=0.7, ax=ax)
ax.axvline(0, color=C_RED, ls="--", lw=1.5)
ax.set_xlabel("Residual (mm)", fontsize=11)
ax.set_title("Residual Distribution", fontsize=12, fontweight="bold")

# ── (B) Q-Q Plot ───────────────────────────────────────────────────
ax2 = axes[1]
(osm, osr), (slope, intercept, r) = probplot(residuals, dist="norm")
ax2.scatter(osm, osr, s=8, color=C_BLUE, alpha=0.6)
x_line = np.linspace(min(osm), max(osm), 100)
ax2.plot(x_line, slope * x_line + intercept, color=C_RED, lw=1.5, ls="--")
ax2.set_xlabel("Theoretical Quantiles", fontsize=11)
ax2.set_ylabel("Sample Quantiles", fontsize=11)
ax2.set_title("Normal Q-Q Plot of Residuals", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()


## 8 · Interpretability

Linear regression offers the rarest gift in machine learning: **coefficients that speak the language of engineering**. Each coefficient tells us, in physical units (mm per unit of input), how much the head diameter changes when one variable increases by one unit — all else held equal.

This transparency is not a consolation prize for using a "simple" model. It is the design goal. A model that an engineer cannot interrogate is a model an engineer will not trust.


In [ ]:
# ── Coefficient table ──────────────────────────────────────────────
coef_df = pd.DataFrame({
    "Feature"    : FEATURES,
    "Coefficient": lr.coef_,
    "Abs_Impact" : np.abs(lr.coef_),
}).sort_values("Abs_Impact", ascending=False).reset_index(drop=True)

coef_df["Direction"] = coef_df["Coefficient"].apply(lambda c: "▲ Increases diameter" if c > 0 else "▼ Decreases diameter")

print(f"OLS Intercept: {lr.intercept_:.4f} mm  (baseline at all-zero inputs)")
display(coef_df[["Feature","Coefficient","Direction"]].style.format({"Coefficient": "{:+.6f}"}))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = [C_RED if c > 0 else C_BLUE for c in coef_df["Coefficient"]]
bars = ax.barh(coef_df["Feature"][::-1], coef_df["Coefficient"][::-1],
               color=colors[::-1], alpha=0.82, edgecolor="white", height=0.6)
ax.axvline(0, color=C_DARK, lw=1.0)

# Value labels
for bar, val in zip(bars, coef_df["Coefficient"][::-1]):
    offset = 0.0002 if val >= 0 else -0.0002
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"{val:+.5f}", va="center", ha=ha, fontsize=9, color=C_DARK)

ax.set_xlabel("Regression Coefficient (mm per unit input)", fontsize=11)
ax.set_title("OLS Coefficients — Effect on Head Diameter per Unit Change",
             fontsize=12, fontweight="bold")
ax.text(0.98, 0.02, "Red: increases diameter  |  Blue: (none in this model)",
        transform=ax.transAxes, fontsize=8, color=C_MUTED, ha="right")
plt.tight_layout()
plt.show()

print("\nEngineering interpretation (top drivers):")
for _, row in coef_df.iterrows():
    print(f"  {row['Feature']:<28}: {row['Coefficient']:+.4f} mm per unit")


In [ ]:
# ── SHAP — Local & global explainability ────────────────────────────
try:
    import shap
    shap.initjs()

    explainer   = shap.Explainer(lr, X_train)
    shap_values = explainer(X_test)

    plt.figure()
    shap.summary_plot(shap_values, X_test, feature_names=FEATURES,
                      plot_type="dot", show=False, max_display=6)
    plt.title("SHAP Summary — Feature Impact on Head Diameter", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("shap not installed — run: pip install shap")
    print("\nFalling back to permutation importance:")
    pi = permutation_importance(lr, X_test, y_test, n_repeats=30, random_state=42)
    pi_df = pd.DataFrame({
        "Feature": FEATURES,
        "Importance_mean": pi.importances_mean,
        "Importance_std" : pi.importances_std,
    }).sort_values("Importance_mean", ascending=False)
    display(pi_df.round(6))


In [ ]:
# ── 2D Response Surface: Press Force × Press Stroke ────────────────
# Fix rivet_diameter_mm = 4.0, rivet_length_mm = 12.0, 
# temperature_c = 24.0, hold_time_ms = 120.0

force_grid  = np.linspace(25, 60, 60)
stroke_grid = np.linspace(3.0, 6.0, 60)
F, S = np.meshgrid(force_grid, stroke_grid)

grid_df = pd.DataFrame({
    "press_force_kn"    : F.ravel(),
    "press_stroke_mm"   : S.ravel(),
    "rivet_diameter_mm" : 4.0,
    "rivet_length_mm"   : 12.0,
    "temperature_c"     : 24.0,
    "hold_time_ms"      : 120.0,
})

Z = lr.predict(grid_df[FEATURES]).reshape(F.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(F, S, Z, levels=25, cmap="RdYlGn_r", alpha=0.85)
plt.colorbar(cf, ax=ax, label="Predicted Head Diameter (mm)")

# Spec window contours
cs = ax.contour(F, S, Z, levels=[SPEC_LO, SPEC_HI],
                colors=["white", "white"], linestyles=["--", "--"], linewidths=1.8)
ax.clabel(cs, fmt="%.1f mm", fontsize=9, colors="white")

# Fill the in-spec zone with a hatch pattern
ax.contourf(F, S, Z, levels=[SPEC_LO, SPEC_HI], colors=["lime"], alpha=0.18, hatches=["////"])

ax.set_xlabel("Press Force (kN)", fontsize=11)
ax.set_ylabel("Press Stroke (mm)", fontsize=11)
ax.set_title("2D Response Surface — Head Diameter as Function of Press Force & Stroke\n"
             "(rivet ∅ 4 mm | length 12 mm | temp 24 °C | hold 120 ms)",
             fontsize=11, fontweight="bold")
ax.text(0.98, 0.03, "Green hatch = Spec window [5.0 – 5.4 mm]",
        transform=ax.transAxes, fontsize=9, color="white", ha="right")
plt.tight_layout()
plt.show()


## 9 · Statistical Validation of Regression Assumptions

A linear regression model is only fully reliable when its **four classical assumptions** hold on the residuals. We test each one formally and visually on the test-set residuals (n = 353).

| Assumption | Test Used | Decision Rule |
|---|---|---|
| **Normality of residuals** | D'Agostino-Pearson | p > 0.05 → residuals are normal |
| **Homoscedasticity** | Goldfeld-Quandt | p > 0.05 → constant variance |
| **No autocorrelation** | Durbin-Watson | DW ≈ 2 → no autocorrelation |
| **No severe multicollinearity** | VIF (Variance Inflation Factor) | VIF < 10 → acceptable |

> **Note on large samples:** With n = 353, even minor departures from normality can reach statistical significance. We prioritise the *practical* interpretation — residuals centred at zero with an RMSE of 0.081 mm represent negligible prediction error relative to the 1.0 mm spec window.


In [ ]:
from scipy.stats import normaltest
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_goldfeldquandt
from statsmodels.tools import add_constant

y_pred_test = lr.predict(X_test)
residuals   = y_test.values - y_pred_test

# ── Normality ────────────────────────────────
dag_stat, dag_p = normaltest(residuals)
print(f"D'Agostino-Pearson normality test:")
print(f"  Stat = {dag_stat:.4f}  |  p-value = {dag_p:.4f}")
print(f"  → {'Residuals appear normal (p > 0.05)' if dag_p > 0.05 else 'Mild departure from normality detected (p ≤ 0.05)'}")
print(f"    Practical note: RMSE = {rmse_test:.4f} mm — error is small relative to spec width.")

# ── Autocorrelation ──────────────────────────
dw = durbin_watson(residuals)
print(f"\nDurbin-Watson autocorrelation test:")
print(f"  DW = {dw:.4f}  (ideal ≈ 2.0)")
print(f"  → {'No significant autocorrelation detected.' if 1.5 < dw < 2.5 else 'Possible autocorrelation — check data ordering.'}")

# ── Homoscedasticity ─────────────────────────
X_test_c = add_constant(X_test.reset_index(drop=True))
gq_stat, gq_p, gq_alt = het_goldfeldquandt(residuals, X_test_c.values)
print(f"\nGoldfeld-Quandt homoscedasticity test:")
print(f"  Stat = {gq_stat:.4f}  |  p-value = {gq_p:.4f}")
print(f"  → {'Constant variance (homoscedastic) — assumption holds.' if gq_p > 0.05 else 'Evidence of heteroscedasticity — consider WLS or variance-stabilising transform.'}")

# ── VIF ──────────────────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor
X_vif = add_constant(X_train.reset_index(drop=True))
vif_data = pd.DataFrame({
    "Feature": ["intercept"] + FEATURES,
    "VIF"    : [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print("\nVariance Inflation Factors (VIF):")
display(vif_data.iloc[1:].reset_index(drop=True).style.format({"VIF": "{:.2f}"}))
print("  → VIF < 5: no multicollinearity concern.")


## 10 · Process Simulator

The model becomes an engineering tool here. Instead of waiting for the CMM to report a non-conformance, the process engineer can query the model *before* the press cycle:  "If I switch to a 5 mm rivet with the same press recipe, what head diameter will I get?"

Three operational scenarios demonstrate the predictive power:

| Scenario | Story | Expected outcome |
|---|---|---|
| **A — Standard operation** | 4 mm rivet, baseline press recipe | In spec |
| **B — Unannounced rivet change** | 5 mm rivet, press unchanged | Out of spec (too large) |
| **C — Engineered adjustment** | 5 mm rivet, press recipe corrected | Back in spec |


In [ ]:
def predict_diameter(press_force_kn, press_stroke_mm, rivet_diameter_mm,
                      rivet_length_mm, temperature_c, hold_time_ms):
    """
    Predict formed rivet head diameter for a given set of process parameters.

    Returns
    -------
    pred_mm : float   — predicted head diameter (mm)
    status  : str     — 'OK' if within [5.0, 5.4] mm, else 'NG'
    margin  : float   — distance to nearest spec limit (mm), negative = outside spec
    """
    x = pd.DataFrame([{
        "press_force_kn"   : press_force_kn,
        "press_stroke_mm"  : press_stroke_mm,
        "rivet_diameter_mm": rivet_diameter_mm,
        "rivet_length_mm"  : rivet_length_mm,
        "temperature_c"    : temperature_c,
        "hold_time_ms"     : hold_time_ms,
    }])
    pred_mm = lr.predict(x)[0]
    status  = "OK" if SPEC_LO <= pred_mm <= SPEC_HI else "NG"
    margin  = min(pred_mm - SPEC_LO, SPEC_HI - pred_mm)
    return pred_mm, status, margin


In [ ]:
# ── Scenario A — Standard 4 mm rivet, baseline press ────────────────
a_pred, a_status, a_margin = predict_diameter(
    press_force_kn=42, press_stroke_mm=4.5,
    rivet_diameter_mm=4.0, rivet_length_mm=12.0,
    temperature_c=24.0, hold_time_ms=120.0
)
print("═══ Scenario A — Standard Operation ════════════════════")
print(f"  Process : 4 mm rivet | Force 42 kN | Stroke 4.5 mm")
print(f"  Predicted head diameter : {a_pred:.3f} mm")
print(f"  Spec window             : [{SPEC_LO}–{SPEC_HI}] mm")
print(f"  Status                  : {a_status} {'✓' if a_status=='OK' else '✗'}")
print(f"  Margin to nearest limit : {a_margin:+.3f} mm")

# ── Scenario B — 5 mm rivet, press recipe unchanged ─────────────────
b_pred, b_status, b_margin = predict_diameter(
    press_force_kn=42, press_stroke_mm=4.5,
    rivet_diameter_mm=5.0, rivet_length_mm=12.0,
    temperature_c=24.0, hold_time_ms=120.0
)
print("\n═══ Scenario B — Unannounced Rivet Batch Change ═════════")
print(f"  Process : 5 mm rivet | Force 42 kN | Stroke 4.5 mm (unchanged)")
print(f"  Predicted head diameter : {b_pred:.3f} mm")
print(f"  Status                  : {b_status} {'✓' if b_status=='OK' else '✗'}")
print(f"  Margin to nearest limit : {b_margin:+.3f} mm")
print(f"  Action needed           : Reduce force and/or stroke before running.")

# ── Scenario C — 5 mm rivet, press recipe corrected ─────────────────
c_pred, c_status, c_margin = predict_diameter(
    press_force_kn=32, press_stroke_mm=3.6,
    rivet_diameter_mm=5.0, rivet_length_mm=12.0,
    temperature_c=24.0, hold_time_ms=120.0
)
print("\n═══ Scenario C — Engineered Press Adjustment ════════════")
print(f"  Process : 5 mm rivet | Force 32 kN | Stroke 3.6 mm (adjusted)")
print(f"  Predicted head diameter : {c_pred:.3f} mm")
print(f"  Status                  : {c_status} {'✓' if c_status=='OK' else '✗'}")
print(f"  Margin to nearest limit : {c_margin:+.3f} mm")
print(f"  Insight: Reducing force by 10 kN and stroke by 0.9 mm")
print(f"           compensates for the larger rivet and restores spec compliance.")


In [ ]:
# ── Optimal Settings Grid Search ────────────────────────────────────
# Given a target diameter and rivet size, find the press settings
# that minimise deviation from the target.

def find_optimal_settings(
    rivet_diameter=4.0, target_mm=SPEC_MID,
    force_range=(25, 60), stroke_range=(3.0, 6.0),
    n_grid=25
):
    forces  = np.linspace(*force_range, n_grid)
    strokes = np.linspace(*stroke_range, n_grid)
    rows = []
    for f in forces:
        for s in strokes:
            pred, status, margin = predict_diameter(
                press_force_kn=f, press_stroke_mm=s,
                rivet_diameter_mm=rivet_diameter, rivet_length_mm=12.0,
                temperature_c=24.0, hold_time_ms=120.0
            )
            rows.append({
                "press_force_kn": round(f, 2),
                "press_stroke_mm": round(s, 2),
                "rivet_diameter_mm": rivet_diameter,
                "predicted_mm": round(pred, 4),
                "status": status,
                "error_abs_mm": round(abs(pred - target_mm), 4),
            })
    df_opt = pd.DataFrame(rows)
    df_opt = df_opt[df_opt["status"] == "OK"].sort_values("error_abs_mm").reset_index(drop=True)
    return df_opt

print("═══ Top 10 Settings for 4 mm Rivet → Target 5.20 mm ════════")
df4 = find_optimal_settings(rivet_diameter=4.0, target_mm=5.20)
display(df4.head(10))

print("\n═══ Top 10 Settings for 5 mm Rivet → Target 5.20 mm ════════")
df5 = find_optimal_settings(rivet_diameter=5.0, target_mm=5.20)
display(df5.head(10))

print("\n═══ Top 10 Settings for 3 mm Rivet → Target 5.20 mm ════════")
df3 = find_optimal_settings(rivet_diameter=3.0, target_mm=5.20)
display(df3.head(10))


## 11 · Final Reflection

---

### What the model tells us

A linear model with six process variables explains **90.9% of the variance** in rivet head diameter, with a typical prediction error of **0.081 mm** — representing just 8.1% of the 1.0 mm specification window. This level of precision is operationally meaningful: it is smaller than most press measurement system resolutions and well below the detection threshold of manual inspection.

The coefficient structure confirms what metallurgical intuition would suggest:

- **Rivet shank diameter** (+0.186 mm/mm) dominates — the amount of material available governs how large the head can be.
- **Press stroke** (+0.075 mm/mm) and **press force** (+0.019 mm/kN) are the primary controllable levers for tuning the diameter in real time.
- **Temperature**, **rivet length**, and **hold time** have small but consistent effects that compound across high-volume production.

### What the model cannot tell us

The model is linear and additive. It captures first-order physics well, but misses nonlinear interactions that emerge at extreme parameter combinations — for example, material fracture thresholds at very high force or cold-working saturation at long hold times. Outside the training envelope [25–60 kN, 3–6 mm stroke], extrapolation should be treated with caution.

It also carries no uncertainty estimate. A single prediction of 5.236 mm does not say "I could be off by ±0.081 mm." A Bayesian regression or conformal prediction interval would add that dimension for high-stakes decisions.

### The operational payoff

The 2D response surface and the optimal-settings search transform the model from a passive estimator into an active process design tool. An engineer facing a batch of 3 mm rivets and a diameter target of 5.2 mm can query the grid and receive a recipe — in seconds, without a physical trial. That is the promise of industrial ML: not to replace engineering judgment, but to give it a faster, more quantitative foundation.

---

*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*
